In [1]:
# Authenticate and initialize
import ee
ee.Authenticate()

# Replace 'your-google-cloud-project-id' with your actual Google Cloud project ID
# You can find your project ID in the Google Cloud Console.
ee.Initialize(project='ee-rubelahmmed01')

import geemap

In [2]:
import os
from google.colab import files

# Define the desired directory path
target_directory = '/content/Shape_File'

uploaded = files.upload(target_directory)

Saving ROI_01.cpg to /content/Shape_File/ROI_01.cpg
Saving ROI_01.dbf to /content/Shape_File/ROI_01.dbf
Saving ROI_01.prj to /content/Shape_File/ROI_01.prj
Saving ROI_01.qix to /content/Shape_File/ROI_01.qix
Saving ROI_01.shp to /content/Shape_File/ROI_01.shp
Saving ROI_01.shx to /content/Shape_File/ROI_01.shx


In [3]:
import geopandas as gpd
import json
# Load shapefile (change filename)
gdf = gpd.read_file("/content/Shape_File/ROI_01.shp")

# Export as GeoJSON
gdf.to_file("/content/Shape_File/ROI_01.geojson", driver="GeoJSON")

# Load GeoJSON
with open("/content/Shape_File/ROI_01.geojson") as f:
    geojson_data = json.load(f)

# Convert to EE Geometry or FeatureCollection
roi = ee.FeatureCollection(geojson_data)

# **Water Mask From Sentinel 2 SCL band**

In [7]:
# Load Sentinel-2 surface reflectance with SCL band
collection = ee.ImageCollection("COPERNICUS/S2_SR") \
      .filterBounds(roi) \
      .filterDate("2024-01-01", "2024-02-01") \
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 70)) \
      .select('SCL')

image = collection.first();  # Choose the first image

# Create map
Map = geemap.Map()
Map.centerObject(roi, zoom=10)

#Define visualization parameters for the SCL band
sclVis = {
    'min': 1,
    'max': 11,
    'palette': [
        '#ff0004', # 1 - Saturated or defective
        '#868686', # 2 - Dark Area Pixels
        '#774b0a', # 3 - Cloud Shadows
        '#10d22c', # 4 - Vegetation
        '#ffff52', # 5 - Bare Soils
        '#0000ff', # 6 - Water
        '#818181', # 7 - Clouds Low Probability / Unclassified
        '#c0c0c0', # 8 - Clouds Medium Probability
        '#f1f1f1', # 9 - Clouds High Probability
        '#bac5eb', # 10 - Cirrus
        '#52fff9'  # 11 - Snow / Ice
    ]
}

#Add the SCL layer to the map
Map.addLayer(image.select('SCL'), sclVis, 'SCL Classified Visualization')
# Correcting the syntax for zoom
Map.centerObject(roi, zoom=10)
Map

Map(center=[25.42137554258588, 89.69539631984811], controls=(WidgetControl(options=['position', 'transparent_b…

# **Water Mask From Sentinel 1 without filter**

In [11]:
# Load Sentinel-1 SAR GRD collection with proper parentheses
s1 = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(roi)
    .filterDate('2024-09-01', '2024-10-01')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))  # Interferometric Wide swath
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))  # or DESCENDING
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('resolution_meters', 10))
    .select('VV')
)

# Get median image and apply speckle filter
image = s1.median().clip(roi)
vv_filtered = image.focal_median(30, 'circle', 'meters')

# Threshold-based water mask
water = vv_filtered.lt(-17)  # adjust threshold if needed

# Add layers to map
Map.addLayer(vv_filtered, {'min': -25, 'max': 0}, 'VV Filtered')
Map.addLayer(water.updateMask(water), {'palette': '0000FF'}, 'Water Mask')

# Show map
Map

Map(bottom=112147.0, center=[25.513939382407116, 89.77066040039064], controls=(WidgetControl(options=['positio…

# **Water Mask From Sentinel 1 without filter** ref BIL